# 03 - Tester la performance des fonctions de detection

Ce notebook ne lit pas de PDF. Il utilise un tableau pandas renseigne directement dans le notebook pour tester les differents detecteurs : Article 9, mots interdits, clauses beneficiaires et conseil.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
elif not (repo_root / "src").exists() and (repo_root.parent / "src").exists():
    repo_root = repo_root.parent

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

repo_root

In [ ]:
from compliance_nlp import load_article9_terms, load_forbidden_terms, load_whitelist_terms
from compliance_nlp.article9 import analyze_article9_section
from compliance_nlp.pipeline import analyze_text
from compliance_nlp.rules import (
    analyze_advice_section,
    analyze_beneficiary_section,
    analyze_forbidden_terms,
)

import pandas as pd

## Chargement des referentiels

Les tests ci-dessous utilisent les fichiers de configuration du projet. Tu peux donc modifier les CSV puis relancer ce notebook.

In [ ]:
forbidden_terms = load_forbidden_terms(repo_root / "configs" / "forbidden_words.csv")
article9_terms = load_article9_terms(repo_root / "configs" / "article9_terms.csv")
whitelist_terms = load_whitelist_terms(repo_root / "configs" / "article9_whitelist.csv")

pd.DataFrame([
    {"referentiel": "forbidden_words", "count": len(forbidden_terms)},
    {"referentiel": "article9_terms", "count": len(article9_terms)},
    {"referentiel": "article9_whitelist", "count": len(whitelist_terms)},
])

## Tableau des cas de test

Modifie ou complete ce tableau. Colonnes utiles :

- `control_family` : `article9`, `forbidden`, `beneficiary`, `advice` ou `full_text`
- `text` : phrase ou bloc texte a tester
- `expected_alert` : `True` si une alerte est attendue
- `expected_keys` : regles attendues separees par `|`. Pour Article 9, utiliser `rule_id`. Sinon utiliser le `code` de finding.
- `expected_category` : categorie Article 9 attendue si utile
- `expected_detection_type` : `exact`, `synonym`, `root`, `fuzzy` si utile

In [ ]:
test_cases = pd.DataFrame([
    {
        "case_id": "A9_EXACT_DIABETE",
        "control_family": "article9",
        "text": "Le client indique etre diabetique depuis plusieurs annees.",
        "expected_alert": True,
        "expected_keys": "SANTE_DIABETE",
        "expected_category": "sante",
        "expected_detection_type": "exact",
        "comment": "Terme exact du dictionnaire Article 9.",
    },
    {
        "case_id": "A9_SYNONYM_INSULINE",
        "control_family": "article9",
        "text": "Observation libre : traitement par insuline signale par l'adherent.",
        "expected_alert": True,
        "expected_keys": "SANTE_DIABETE",
        "expected_category": "sante",
        "expected_detection_type": "synonym",
        "comment": "Synonyme parametre.",
    },
    {
        "case_id": "A9_FUZZY_DIABETTIQUE",
        "control_family": "article9",
        "text": "Le souscripteur precise etre diabettique.",
        "expected_alert": True,
        "expected_keys": "SANTE_DIABETE",
        "expected_category": "sante",
        "expected_detection_type": "fuzzy",
        "comment": "Faute volontaire sur diabetique.",
    },
    {
        "case_id": "A9_WHITELIST_SANTE_FINANCIERE",
        "control_family": "article9",
        "text": "La sante financiere du contrat est correcte.",
        "expected_alert": False,
        "expected_keys": "",
        "expected_category": "",
        "expected_detection_type": "",
        "comment": "Expression attendue en whitelist.",
    },
    {
        "case_id": "FORBIDDEN_SANS_RISQUE",
        "control_family": "forbidden",
        "text": "Ce placement est sans risque selon le conseiller.",
        "expected_alert": True,
        "expected_keys": "forbidden_term_interdit",
        "expected_category": "",
        "expected_detection_type": "",
        "comment": "Mot interdit configure dans forbidden_words.csv.",
    },
    {
        "case_id": "BENEF_IMPRECISE",
        "control_family": "beneficiary",
        "text": "Je designe la personne qui sera la plus presente pour moi au moment venu.",
        "expected_alert": True,
        "expected_keys": "beneficiary_clause_imprecise",
        "expected_category": "",
        "expected_detection_type": "",
        "comment": "Clause beneficiaire imprecise.",
    },
    {
        "case_id": "ADVICE_RISK_MINIMIZATION",
        "control_family": "advice",
        "text": "Les risques ne sont pas un vrai sujet ici.",
        "expected_alert": True,
        "expected_keys": "advice_risk_minimization",
        "expected_category": "",
        "expected_detection_type": "",
        "comment": "Minimisation du risque dans la zone conseil.",
    },
    {
        "case_id": "NEGATIVE_NEUTRAL",
        "control_family": "article9",
        "text": "Le client souhaite modifier son adresse postale et son mode de versement.",
        "expected_alert": False,
        "expected_keys": "",
        "expected_category": "",
        "expected_detection_type": "",
        "comment": "Phrase neutre sans donnee sensible attendue.",
    },
])

test_cases

## Fonctions d'execution des tests

In [ ]:
def split_expected(value):
    if pd.isna(value) or not str(value).strip():
        return set()
    return {item.strip() for item in str(value).split("|") if item.strip()}


def finding_key(finding):
    return finding.rule_id or finding.code


def run_detector(row):
    family = row["control_family"]
    text = row["text"]

    if family == "article9":
        return analyze_article9_section("test", text, article9_terms, whitelist_terms)
    if family == "forbidden":
        return analyze_forbidden_terms("test", text, forbidden_terms)
    if family == "beneficiary":
        return analyze_beneficiary_section(text)
    if family == "advice":
        return analyze_advice_section(text)
    if family == "full_text":
        return analyze_text(
            document_name=row["case_id"],
            source_path="notebook",
            extracted_text=text,
            forbidden_terms=forbidden_terms,
            article9_terms=article9_terms,
            whitelist_terms=whitelist_terms,
        ).findings

    raise ValueError(f"control_family inconnu: {family}")


def findings_to_rows(case, findings):
    if not findings:
        return [{
            "case_id": case["case_id"],
            "control_family": case["control_family"],
            "predicted_key": None,
            "code": None,
            "rule_id": None,
            "category": None,
            "matched_term": None,
            "detection_type": None,
            "score": None,
            "severity": None,
            "evidence": None,
        }]

    return [
        {
            "case_id": case["case_id"],
            "control_family": case["control_family"],
            "predicted_key": finding_key(finding),
            "code": finding.code,
            "rule_id": finding.rule_id,
            "category": finding.category,
            "matched_term": finding.matched_term,
            "detection_type": finding.detection_type,
            "score": finding.score,
            "severity": finding.severity,
            "evidence": finding.evidence,
        }
        for finding in findings
    ]

## Lancement du banc de test

In [ ]:
case_results = []
finding_rows = []

for _, case in test_cases.iterrows():
    findings = run_detector(case)
    predicted_keys = {finding_key(finding) for finding in findings}
    expected_keys = split_expected(case["expected_keys"])
    detected = bool(findings)
    expected_alert = bool(case["expected_alert"])

    if expected_alert and detected:
        outcome = "TP"
    elif expected_alert and not detected:
        outcome = "FN"
    elif not expected_alert and detected:
        outcome = "FP"
    else:
        outcome = "TN"

    expected_key_match = not expected_keys or bool(expected_keys & predicted_keys)
    max_score = max([finding.score for finding in findings if finding.score is not None], default=None)

    case_results.append({
        **case.to_dict(),
        "detected": detected,
        "predicted_keys": " | ".join(sorted(predicted_keys)),
        "finding_count": len(findings),
        "max_score": max_score,
        "outcome": outcome,
        "expected_key_match": expected_key_match,
    })
    finding_rows.extend(findings_to_rows(case, findings))

case_results_df = pd.DataFrame(case_results)
findings_df = pd.DataFrame(finding_rows)

case_results_df

## Detail des findings produits

In [ ]:
findings_df.sort_values(["case_id", "score"], ascending=[True, False], na_position="last")

## Mesures globales de performance

In [ ]:
tp = (case_results_df["outcome"] == "TP").sum()
fp = (case_results_df["outcome"] == "FP").sum()
fn = (case_results_df["outcome"] == "FN").sum()
tn = (case_results_df["outcome"] == "TN").sum()

precision = tp / (tp + fp) if (tp + fp) else None
recall = tp / (tp + fn) if (tp + fn) else None
f1 = 2 * precision * recall / (precision + recall) if precision and recall else None

pd.DataFrame([{
    "TP": tp,
    "FP": fp,
    "FN": fn,
    "TN": tn,
    "precision": precision,
    "recall": recall,
    "f1": f1,
}])

In [ ]:
case_results_df.groupby(["control_family", "outcome"], dropna=False).size().reset_index(name="count")

## Faux positifs, faux negatifs et erreurs de regle

In [ ]:
case_results_df[
    (case_results_df["outcome"].isin(["FP", "FN"]))
    | ((case_results_df["expected_alert"]) & (~case_results_df["expected_key_match"]))
][[
    "case_id",
    "control_family",
    "text",
    "expected_alert",
    "expected_keys",
    "predicted_keys",
    "outcome",
    "expected_key_match",
    "max_score",
    "comment",
]]

## Analyse Article 9 par type de detection

In [ ]:
article9_findings = findings_df[findings_df["control_family"].isin(["article9", "full_text"])].copy()
article9_findings[article9_findings["predicted_key"].notna()].groupby(
    ["category", "detection_type"], dropna=False
).agg(
    count=("predicted_key", "size"),
    avg_score=("score", "mean"),
    min_score=("score", "min"),
    max_score=("score", "max"),
).reset_index().sort_values(["category", "detection_type"])

## Simulation des seuils de score Article 9

In [ ]:
article9_cases = case_results_df[case_results_df["control_family"].isin(["article9", "full_text"])].copy()
threshold_rows = []

for threshold in [0.50, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]:
    evaluated = article9_cases.copy()
    evaluated["detected_at_threshold"] = evaluated["max_score"].fillna(0) >= threshold

    tp_t = ((evaluated["expected_alert"]) & (evaluated["detected_at_threshold"])).sum()
    fp_t = ((~evaluated["expected_alert"]) & (evaluated["detected_at_threshold"])).sum()
    fn_t = ((evaluated["expected_alert"]) & (~evaluated["detected_at_threshold"])).sum()
    tn_t = ((~evaluated["expected_alert"]) & (~evaluated["detected_at_threshold"])).sum()
    precision_t = tp_t / (tp_t + fp_t) if (tp_t + fp_t) else None
    recall_t = tp_t / (tp_t + fn_t) if (tp_t + fn_t) else None
    f1_t = 2 * precision_t * recall_t / (precision_t + recall_t) if precision_t and recall_t else None

    threshold_rows.append({
        "threshold": threshold,
        "TP": tp_t,
        "FP": fp_t,
        "FN": fn_t,
        "TN": tn_t,
        "precision": precision_t,
        "recall": recall_t,
        "f1": f1_t,
    })

thresholds_df = pd.DataFrame(threshold_rows)
thresholds_df

## Export des resultats du banc de test

In [ ]:
export_dir = repo_root / "outputs" / "performance"
export_dir.mkdir(parents=True, exist_ok=True)

case_results_path = export_dir / "detection_case_results.csv"
findings_path = export_dir / "detection_findings.csv"
thresholds_path = export_dir / "article9_thresholds.csv"

case_results_df.to_csv(case_results_path, index=False, encoding="utf-8-sig")
findings_df.to_csv(findings_path, index=False, encoding="utf-8-sig")
thresholds_df.to_csv(thresholds_path, index=False, encoding="utf-8-sig")

{
    "case_results": str(case_results_path),
    "findings": str(findings_path),
    "thresholds": str(thresholds_path),
}